In [1]:
from google.colab import drive
from torchvision.datasets import ImageFolder
from torchvision import transforms
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
train_transform = transforms.Compose([transforms.Resize((64, 64)),
                                      transforms.RandomRotation(10),
                                      transforms.RandomHorizontalFlip(),
                                      transforms.ToTensor(),
                                      transforms.Normalize((0.5, 0.5, 0.5),
                                                           (0.5, 0.5, 0.5))
                                      ])

test_transform = transforms.Compose([transforms.Resize((64, 64)),
                                     transforms.ToTensor(),
                                     transforms.Normalize((0.5, 0.5, 0.5),
                                                          (0.5, 0.5, 0.5))])

In [4]:
!cp -r "/content/drive/MyDrive/ML DATASETS/CatsVsDogs" "/content/"

In [5]:
train_data = ImageFolder(root="/content/CatsVsDogs/dataset/train", transform=train_transform)
validation_data = ImageFolder(root="/content/CatsVsDogs/dataset/validation", transform=test_transform)
test_data = ImageFolder(root="//content/CatsVsDogs/dataset/test", transform=test_transform)

In [6]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
validation_loader = DataLoader(validation_data, batch_size=64, shuffle=False, num_workers=2)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=2)

In [13]:
class CatDog(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(3, 32, 3)

    self.conv2 = nn.Conv2d(32, 64, 3)

    self.conv3 = nn.Conv2d(64, 128, 3)

    self.maxpool = nn.MaxPool2d(2, 2)
    self.dropout = nn.Dropout(0.3)
    self.fc1 = nn.Linear(4608, 1)


  def forward(self, x):
    x = self.conv1(x)
    x = torch.relu(x)
    x = self.maxpool(x)
    #print('shape after conv1: ', x.shape)

    x = self.conv2(x)
    x = torch.relu(x)
    x = self.maxpool(x)
    #print('shape after conv2', x.shape)

    x = self.conv3(x)
    x = torch.relu(x)
    x = self.maxpool(x)
    #print('shape after conv3', x.shape)

    x = x.view(x.size(0), -1)
    x = self.dropout(x)
    x = self.fc1(x)
    #print('shape after flattening', x.shape)
    return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#model = CatDog().to(device)
#dummy = torch.randn(64, 3, 64, 64).to(device)
#out = model(dummy)
#loss_fn = nn.BCEWithLogitsLoss()
#optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [14]:
model = CatDog().to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [9]:
print(torch.cuda.is_available())
print(next(model.parameters()).is_cuda)

True
True


In [15]:
for epochs in range(15):
  for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)
    optimizer.zero_grad()
    output = model(images)
    labels = labels.float().unsqueeze(1)
    loss = loss_fn(output, labels)
    loss.backward()
    optimizer.step()
  print(f'Epoch: {epochs+1} | Loss: {loss.item():.4f}')

Epoch: 1 | Loss: 0.5609
Epoch: 2 | Loss: 0.3196
Epoch: 3 | Loss: 0.6002
Epoch: 4 | Loss: 0.2835
Epoch: 5 | Loss: 0.1841
Epoch: 6 | Loss: 0.2502
Epoch: 7 | Loss: 0.2239
Epoch: 8 | Loss: 0.2663
Epoch: 9 | Loss: 0.2447
Epoch: 10 | Loss: 0.3348
Epoch: 11 | Loss: 0.3877
Epoch: 12 | Loss: 0.3040
Epoch: 13 | Loss: 0.3288
Epoch: 14 | Loss: 0.1701
Epoch: 15 | Loss: 0.2381


In [20]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
  for inputs, labels in validation_loader:
    inputs = inputs.to(device)
    labels = labels.float().unsqueeze(1).to(device)
    outputs = model(inputs)
    prob = torch.sigmoid(outputs)
    pred = (prob > 0.5).float()
    total += labels.size(0)
    correct += (pred == labels).sum().item()

accuracy = correct / total
print(f'val acc: {accuracy:.4f}')

val acc: 0.8892
